Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


PCA init...


Epoch 1: 100%|██████████| 10/10 [00:01<00:00,  5.63it/s]


Epoch 1  Loss:2.3547
Val Acc: 0.1500


Epoch 2: 100%|██████████| 10/10 [00:01<00:00,  7.30it/s]


...

Epoch 21: 100%|██████████| 10/10 [00:02<00:00,  4.30it/s]


Epoch 21  Loss:2.2884
Val Acc: 0.2000


Epoch 22: 100%|██████████| 10/10 [00:02<00:00,  4.31it/s]


Epoch 22  Loss:2.2872
Val Acc: 0.1750
Early stop
Test Acc: 0.1800


0.18

In [64]:
!pip install -q gpytorch transformers tqdm scikit-learn torchvision

import json
import math
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from PIL import Image
import gpytorch
from transformers import ViTForImageClassification, ViTConfig, AutoImageProcessor
from torchvision import transforms
import tqdm
from sklearn.cluster import MiniBatchKMeans
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import ExponentialLR

# === CONFIG ===
pretrained_vit_dir = '/kaggle/input/models/pretrained_vit/pretrained_vit' 
eurosat_input = '/kaggle/input/eurosat-dataset/EuroSAT'
num_classes = 10
low_dim = 10
batch_size = 16
n_epochs = 100
lr_vit = 1e-8
lr_gp = 1e-3
patience = 10
test_size_val = 0.2
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = True
WARMUP_EPOCHS = 5

# === DATA ===
train_df = pd.read_csv(f'{eurosat_input}/train.csv')
test_df = pd.read_csv(f'{eurosat_input}/test.csv')
classes = sorted(train_df['Label'].unique())
label_map = {cls: i for i, cls in enumerate(classes)}

train_subset_df = pd.DataFrame()
for label in classes:
    class_df = train_df[train_df['Label'] == label]
    sampled = class_df.sample(n=20, random_state=42)
    train_subset_df = pd.concat([train_subset_df, sampled])

image_dir = eurosat_input + '/'
train_subset_df['image_path'] = image_dir + train_subset_df['Filename']
test_df['image_path'] = image_dir + test_df['Filename']

train_sub, val_sub = train_test_split(train_subset_df, test_size=test_size_val, stratify=train_subset_df['Label'], random_state=42)
train_sub['label'] = train_sub['Label'].map(label_map)
val_sub['label'] = val_sub['Label'].map(label_map)
test_df['label'] = test_df['Label'].map(label_map)

print(f"Train: {train_sub.shape} | Val: {val_sub.shape} | Test: {test_df.shape}")

# === DATASET ===
processor = AutoImageProcessor.from_pretrained('google/vit-base-patch16-224')
aug_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(90),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
])

class EuroSATDataset(Dataset):
    def __init__(self, df, processor, augment=False):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.augment = augment
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['image_path']
        label = self.df.iloc[idx]['label']
        image = Image.open(img_path).convert('RGB')
        if self.augment: image = aug_transform(image)
        inputs = self.processor(image, return_tensors='pt')
        return inputs['pixel_values'].squeeze(0), label

train_dataset = EuroSATDataset(train_sub, processor, augment=True)
val_dataset = EuroSATDataset(val_sub, processor)
test_dataset = EuroSATDataset(test_df, processor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
num_train = len(train_dataset)

# === ViT Feature Extractor ===
config = ViTConfig.from_pretrained(pretrained_vit_dir)
vit_model = ViTForImageClassification.from_pretrained(pretrained_vit_dir, config=config)
feature_extractor = vit_model.vit.to(device)
for param in feature_extractor.parameters():
    param.requires_grad = False

# === Projection + PCA Init ===
scaler = GradScaler(enabled=use_amp)
projection = nn.Linear(768, low_dim).to(device)

print("Extracting features for PCA init...")
train_features = []
with torch.no_grad():
    for data, _ in train_loader:
        data = data.to(device)
        with autocast('cuda', enabled=use_amp):
            out = feature_extractor(data)
            feat = out.pooler_output if out.pooler_output is not None else out.last_hidden_state[:, 0]
            train_features.append(feat.cpu())
train_features = torch.cat(train_features, 0)

# PCA
mean = train_features.mean(0, keepdim=True)
train_features_center = train_features - mean
U, S, Vh = torch.linalg.svd(train_features_center, full_matrices=False)
with torch.no_grad():
    projection.weight.data = Vh[:low_dim].to(device)
    projection.bias.data = (-mean.to(device) @ projection.weight.data.T).squeeze(0)

# Project
projected_features = []
with torch.no_grad():
    for i in range(0, len(train_features), batch_size):
        batch = train_features[i:i+batch_size].to(device)
        projected_features.append(projection(batch).cpu())
projected_features = torch.cat(projected_features, 0).numpy()

# === Inducing Points ===
num_inducing = min(30, num_train)
kmeans = MiniBatchKMeans(n_clusters=num_inducing, n_init='auto', random_state=42)
kmeans.fit(projected_features)
inducing_points = torch.tensor(kmeans.cluster_centers_, dtype=torch.float32).to(device)
inducing_points += 1e-5 * torch.randn_like(inducing_points)

# === GP Layer: RBF WITH HARD LOCKED LENGTHSCALE ===
class GPLayer(gpytorch.models.ApproximateGP):
    def __init__(self, inducing_points, num_classes, input_dim):
        inducing_points = inducing_points + 1e-5 * torch.randn_like(inducing_points)
        variational_distribution = gpytorch.variational.CholeskyVariationalDistribution(
            inducing_points.size(0), batch_shape=torch.Size([num_classes])
        )
        variational_strategy = gpytorch.variational.IndependentMultitaskVariationalStrategy(
            gpytorch.variational.VariationalStrategy(
                self, inducing_points, variational_distribution, learn_inducing_locations=True
            ),
            num_tasks=num_classes
        )
        super().__init__(variational_strategy)

        self.mean_module = gpytorch.means.ConstantMean(batch_shape=torch.Size([num_classes]))

        base_kernel = gpytorch.kernels.RBFKernel(
            ard_num_dims=input_dim,
            batch_shape=torch.Size([num_classes])
        )

        # HARD LENGTHSCALE LOCK
        with torch.no_grad():
            dists = torch.cdist(inducing_points[0], inducing_points[0])
            median_dist = torch.median(dists[dists > 0]).item()
            lengthscale = median_dist * 0.7
            base_kernel.lengthscale = lengthscale

        base_kernel.lengthscale_constraint = gpytorch.constraints.Interval(
            lengthscale * 0.9, lengthscale * 1.1
        )

        self.covar_module = gpytorch.kernels.ScaleKernel(
            base_kernel,
            batch_shape=torch.Size([num_classes]),
            outputscale_constraint=gpytorch.constraints.Interval(0.5, 3.0)
        )

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

# === DKL Model ===
class DKLModel(nn.Module):
    def __init__(self, feature_extractor, low_dim, num_classes, inducing_points):
        super().__init__()
        self.feature_extractor = feature_extractor
        self.projection = projection
        self.gp_layer = GPLayer(inducing_points, num_classes, low_dim)
    def forward(self, x):
        out = self.feature_extractor(x)
        feat = out.pooler_output if out.pooler_output is not None else out.last_hidden_state[:, 0]
        projected = self.projection(feat)
        projected = (projected - projected.mean(0, keepdim=True)) / (projected.std(0, keepdim=True) + 1e-5)
        return self.gp_layer(projected.float())

# === Model + Likelihood ===
model = DKLModel(feature_extractor, low_dim, num_classes, inducing_points).to(device).float()
likelihood = gpytorch.likelihoods.SoftmaxLikelihood(num_classes=num_classes, num_features=num_classes).to(device).float()

# === Optimizer ===
optimizer = AdamW([
    {'params': model.projection.parameters(), 'lr': lr_gp},
    {'params': model.gp_layer.hyperparameters(), 'lr': lr_gp},
    {'params': model.gp_layer.variational_parameters(), 'lr': lr_gp},
    {'params': likelihood.parameters(), 'lr': lr_gp}
])
scheduler = ExponentialLR(optimizer, gamma=0.99)
mll = gpytorch.mlls.VariationalELBO(likelihood, model.gp_layer, num_data=num_train)

# === Training / Eval ===
def train_epoch(epoch):
    model.train()
    likelihood.train()
    total_loss = 0
    for data, target in tqdm.tqdm(train_loader, desc=f'Epoch {epoch}'):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        with autocast('cuda', enabled=use_amp), gpytorch.settings.cholesky_jitter(1e-1):
            output = model(data)
            loss = -mll(output, target)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(train_loader)

def evaluate(loader, split='Val'):
    model.eval()
    likelihood.eval()
    correct = total = 0
    with torch.no_grad(), gpytorch.settings.fast_pred_var(), gpytorch.settings.num_likelihood_samples(64), gpytorch.settings.cholesky_jitter(1e-1):
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = likelihood(model(data))
            pred = output.probs.mean(0).argmax(-1)
            correct += pred.eq(target).sum().item()
            total += target.size(0)
    acc = correct / total
    print(f'{split} Acc: {acc:.4f}')
    return acc

# === Training Loop ===
best_val_acc = 0
patience_counter = 0
for epoch in range(1, n_epochs + 1):
    if epoch == WARMUP_EPOCHS + 1:
        print(f"--- Epoch {epoch}: UNFREEZING ViT ---")
        for param in model.feature_extractor.parameters():
            param.requires_grad = True
        optimizer = AdamW([
            {'params': model.feature_extractor.parameters(), 'lr': lr_vit},
            {'params': model.projection.parameters(), 'lr': lr_gp},
            {'params': model.gp_layer.hyperparameters(), 'lr': lr_gp},
            {'params': model.gp_layer.variational_parameters(), 'lr': lr_gp},
            {'params': likelihood.parameters(), 'lr': lr_gp}
        ])
        scheduler = ExponentialLR(optimizer, gamma=0.99)
    
    avg_loss = train_epoch(epoch)
    scheduler.step()
    print(f'Epoch {epoch}, Avg Loss: {avg_loss:.4f}')
    val_acc = evaluate(val_loader)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({'model': model.state_dict(), 'likelihood': likelihood.state_dict()}, 'best_dkl.pt')
        patience_counter = 0
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print('Early stopping')
        break

# === Final Test ===
checkpoint = torch.load('best_dkl.pt', map_location=device)
model.load_state_dict(checkpoint['model'])
likelihood.load_state_dict(checkpoint['likelihood'])
evaluate(test_loader, 'Test')

Train: (160, 6) | Val: (40, 6) | Test: (2700, 6)


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


Extracting features for PCA init...


RuntimeError: cdist only supports at least 2D tensors, X1 got: 1D

In [37]:
!pip install -q gpytorch transformers tqdm scikit-learn torchvision

In [38]:
import json
import math
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from PIL import Image
import gpytorch
from transformers import ViTForImageClassification, ViTConfig, AutoImageProcessor
from torchvision import transforms
import tqdm
from sklearn.cluster import MiniBatchKMeans
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import ExponentialLR

In [39]:
pretrained_vit_dir = '/kaggle/input/models/pretrained_vit/pretrained_vit' 
eurosat_input = '/kaggle/input/eurosat-dataset/EuroSAT'
num_classes = 10
low_dim = 10
batch_size = 16
n_epochs = 100
lr_vit = 1e-8
lr_gp = 1e-3  # Reduced for stability
patience = 10
test_size_val = 0.2
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = True
WARMUP_EPOCHS = 5

In [40]:
train_df = pd.read_csv(f'{eurosat_input}/train.csv')
test_df = pd.read_csv(f'{eurosat_input}/test.csv')

In [41]:
classes = sorted(train_df['Label'].unique())
label_map = {cls: i for i, cls in enumerate(classes)}

In [42]:
train_subset_df = pd.DataFrame()
for label in classes:
    class_df = train_df[train_df['Label'] == label]
    sampled = class_df.sample(n=20, random_state=42)
    train_subset_df = pd.concat([train_subset_df, sampled])

In [43]:
image_dir = eurosat_input + '/'
train_subset_df['image_path'] = image_dir + train_subset_df['Filename']
test_df['image_path'] = image_dir + test_df['Filename']

In [44]:
train_sub, val_sub = train_test_split(train_subset_df, test_size=test_size_val, stratify=train_subset_df['Label'], random_state=42)

In [45]:
train_sub['label'] = train_sub['Label'].map(label_map)
val_sub['label'] = val_sub['Label'].map(label_map)
test_df['label'] = test_df['Label'].map(label_map)

In [46]:
print(f"Train: {train_sub.shape}")
print(f"Val: {val_sub.shape}")
print(f"Test: {test_df.shape}")

Train: (160, 6)
Val: (40, 6)
Test: (2700, 6)


In [47]:
processor = AutoImageProcessor.from_pretrained('google/vit-base-patch16-224')
aug_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(90),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
])

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


In [48]:
class EuroSATDataset(Dataset):
    def __init__(self, df, processor, augment=False):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['image_path']
        label = self.df.iloc[idx]['label']
        image = Image.open(img_path).convert('RGB')
        if self.augment:
            image = aug_transform(image)
        inputs = self.processor(image, return_tensors='pt')
        pixel_values = inputs['pixel_values'].squeeze(0)
        return pixel_values, label

In [49]:
train_dataset = EuroSATDataset(train_sub, processor, augment=True)
val_dataset = EuroSATDataset(val_sub, processor)
test_dataset = EuroSATDataset(test_df, processor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
num_train = len(train_dataset)

In [50]:
config = ViTConfig.from_pretrained(pretrained_vit_dir)
vit_model = ViTForImageClassification.from_pretrained(pretrained_vit_dir, config=config)
feature_extractor = vit_model.vit.to(device)
for param in feature_extractor.parameters():
    param.requires_grad = False

In [51]:
scaler = GradScaler(enabled=use_amp)
projection = nn.Linear(768, low_dim).to(device)  # Global projection
print("Extracting and PCA-initializing features...")
train_features = []
with torch.no_grad():
    for data, _ in train_loader:
        data = data.to(device)
        with autocast('cuda', enabled=use_amp):
            out = feature_extractor(data)
            feat = out.pooler_output if out.pooler_output is not None else out.last_hidden_state[:, 0]
            train_features.append(feat.cpu())
train_features = torch.cat(train_features, 0)

# PCA
mean = train_features.mean(0, keepdim=True)
train_features_center = train_features - mean
U, S, Vh = torch.linalg.svd(train_features_center, full_matrices=False)

# Initialize projection
with torch.no_grad():
    projection.weight.data = Vh[:low_dim].to(device)
    projection.bias.data = (-mean.to(device) @ projection.weight.data.T).squeeze(0)

# Project features
projected_features = []
with torch.no_grad():
    for i in range(0, len(train_features), batch_size):
        batch = train_features[i:i+batch_size].to(device)
        projected_features.append(projection(batch).cpu())
projected_features = torch.cat(projected_features, 0).numpy()

Extracting and PCA-initializing features...


In [52]:
num_inducing = min(50, num_train)
kmeans = MiniBatchKMeans(n_clusters=num_inducing, n_init='auto', random_state=42)
kmeans.fit(projected_features)

inducing_points = torch.tensor(kmeans.cluster_centers_, dtype=torch.float32)
inducing_points = (inducing_points - inducing_points.mean(0)) / (inducing_points.std(0) + 1e-6)
inducing_points += 1e-4 * torch.randn_like(inducing_points)
inducing_points = inducing_points.to(device)

In [53]:
class GPLayer(gpytorch.models.ApproximateGP):
    def __init__(self, inducing_points, num_classes, input_dim):
        inducing_points = inducing_points + 1e-4 * torch.randn_like(inducing_points)
        variational_distribution = gpytorch.variational.CholeskyVariationalDistribution(
            inducing_points.size(0), batch_shape=torch.Size([num_classes])
        )
        variational_strategy = gpytorch.variational.IndependentMultitaskVariationalStrategy(
            gpytorch.variational.VariationalStrategy(
                self, inducing_points, variational_distribution, learn_inducing_locations=True
            ),
            num_tasks=num_classes
        )
        super().__init__(variational_strategy)

        self.mean_module = gpytorch.means.ConstantMean(batch_shape=torch.Size([num_classes]))
        base_kernel = gpytorch.kernels.LinearKernel(
            ard_num_dims=input_dim,
            batch_shape=torch.Size([num_classes])
        )
        base_kernel.raw_variance_constraint = gpytorch.constraints.Positive()
        self.covar_module = gpytorch.kernels.ScaleKernel(
            base_kernel,
            batch_shape=torch.Size([num_classes]),
            outputscale_constraint=gpytorch.constraints.Interval(1e-3, 10.0)
        )

        self.covar_module.base_kernel.register_prior(
            "variance_prior", gpytorch.priors.NormalPrior(1.0, 1.0), lambda m: m.variance
        )
        self.covar_module.register_prior(
            "outputscale_prior", gpytorch.priors.NormalPrior(1.0, 0.5), lambda m: m.outputscale
        )

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

In [54]:
class DKLModel(nn.Module):
    def __init__(self, feature_extractor, low_dim, num_classes, inducing_points):
        super().__init__()
        self.feature_extractor = feature_extractor
        # <<<----- ADD THIS LINE (the global projection is now an attribute) ----->>>
        self.projection = projection          # <-- THIS WAS MISSING
        # -----------------------------------------------------------------------
        self.gp_layer = GPLayer(inducing_points, num_classes, low_dim)

    def forward(self, x):
        out = self.feature_extractor(x)
        feat = out.pooler_output if out.pooler_output is not None else out.last_hidden_state[:, 0]
        projected = self.projection(feat)                     # <-- now works
        projected = (projected - projected.mean(0, keepdim=True)) / \
                    (projected.std(0, keepdim=True) + 1e-6)
        return self.gp_layer(projected.float())

In [55]:
model = DKLModel(feature_extractor, low_dim, num_classes, inducing_points).to(device).float()
likelihood = gpytorch.likelihoods.SoftmaxLikelihood(num_classes=num_classes, num_features=num_classes).to(device).float()

In [56]:
optimizer = AdamW([
    {'params': model.projection.parameters(), 'lr': lr_gp},
    {'params': model.gp_layer.hyperparameters(), 'lr': lr_gp},
    {'params': model.gp_layer.variational_parameters(), 'lr': lr_gp},
    {'params': likelihood.parameters(), 'lr': lr_gp}
])
scheduler = ExponentialLR(optimizer, gamma=0.99)

In [57]:
mll = gpytorch.mlls.VariationalELBO(likelihood, model.gp_layer, num_data=num_train)

In [58]:
def train_epoch(epoch):
    model.train()
    likelihood.train()
    total_loss = 0
    for data, target in tqdm.tqdm(train_loader, desc=f'Epoch {epoch}'):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        with autocast('cuda', enabled=use_amp), gpytorch.settings.cholesky_jitter(1e-3):
            output = model(data)
            loss = -mll(output, target)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(train_loader)

In [59]:
def evaluate(loader, split='Val'):
    model.eval()
    likelihood.eval()
    correct = total = 0
    with torch.no_grad(), \
         gpytorch.settings.fast_pred_var(), \
         gpytorch.settings.num_likelihood_samples(64), \
         gpytorch.settings.cholesky_jitter(1e-3):
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = likelihood(model(data))
            pred = output.probs.mean(0).argmax(-1)
            correct += pred.eq(target).sum().item()
            total += target.size(0)
    acc = correct / total
    print(f'{split} Acc: {acc:.4f}')
    return acc

In [60]:
best_val_acc = 0
patience_counter = 0
for epoch in range(1, n_epochs + 1):
    if epoch == WARMUP_EPOCHS + 1:
        print(f"--- Epoch {epoch}: STARTING JOINT DKL TRAINING (ViT Unfrozen) ---")
        for param in model.feature_extractor.parameters():
            param.requires_grad = True
        optimizer = AdamW([
            {'params': model.feature_extractor.parameters(), 'lr': lr_vit},
            {'params': model.projection.parameters(), 'lr': lr_gp},
            {'params': model.gp_layer.hyperparameters(), 'lr': lr_gp},
            {'params': model.gp_layer.variational_parameters(), 'lr': lr_gp},
            {'params': likelihood.parameters(), 'lr': lr_gp}
        ])
        scheduler = ExponentialLR(optimizer, gamma=0.99)
    
    avg_loss = train_epoch(epoch)
    scheduler.step()
    print(f'Epoch {epoch}, Avg Loss: {avg_loss:.4f}')
    val_acc = evaluate(val_loader)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({'model': model.state_dict(), 'likelihood': likelihood.state_dict()}, 'best_dkl.pt')
        patience_counter = 0
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print('Early stopping')
        break

Epoch 1:   0%|          | 0/10 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-07 to the diagonal
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-06 to the diagonal
  warnings.warn(
Epoch 1:   0%|          | 0/10 [00:00<?, ?it/s]


NotPSDError: Matrix not positive definite after repeatedly adding jitter up to 1.0e-06.

In [ ]:
checkpoint = torch.load('best_dkl.pt', map_location=device)
model.load_state_dict(checkpoint['model'])
likelihood.load_state_dict(checkpoint['likelihood'])
evaluate(test_loader, 'Test')